# Residual Diagnostics

## Overview

Residual analysis is the primary tool for checking whether a model's assumptions hold. No summary statistic substitutes for plotting residuals — patterns that indicate assumption violations are often invisible in the model summary but immediately obvious in diagnostic plots.

**Four core diagnostic plots:**

| Plot | Checks |
|---|---|
| Residuals vs Fitted | Linearity, homoscedasticity |
| Q-Q plot of residuals | Normality of residuals |
| Scale-Location (sqrt\|resid\| vs fitted) | Homoscedasticity |
| Residuals vs Leverage | Influential observations |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import OLSInfluence

rng = np.random.default_rng(42)
n = 120
elevation = rng.uniform(50, 400, n)
nitrate   = rng.gamma(2, 2, n)
treatment = rng.choice([0,1], n)
richness  = 28 - 0.035*elevation - 0.8*nitrate + 2.5*treatment + rng.normal(0, 3.5, n)
X = sm.add_constant(pd.DataFrame({"elevation":elevation,"nitrate":nitrate,"treatment":treatment}))
model = sm.OLS(richness, X).fit()
print(f"R-squared: {model.rsquared:.3f}, n={n}")

---
## The Four Diagnostic Plots

In [ ]:
resid     = model.resid
fitted    = model.fittedvalues
influence = OLSInfluence(model)
std_resid = influence.resid_studentized_internal
leverage  = influence.hat_matrix_diag
cooks_d   = influence.cooks_distance[0]

fig, axes = plt.subplots(2,2,figsize=(12,9))
# 1. Residuals vs Fitted
axes[0,0].scatter(fitted, resid, alpha=0.5, s=20, color="steelblue")
axes[0,0].axhline(0, color="#e74c3c", lw=1.5)
axes[0,0].set_xlabel("Fitted values"); axes[0,0].set_ylabel("Residuals")
axes[0,0].set_title("Residuals vs Fitted")
# 2. Q-Q
(osm,osr),(sl,ic,r) = stats.probplot(resid, dist="norm")
axes[0,1].scatter(osm, osr, s=20, alpha=0.5, color="steelblue")
axes[0,1].plot(osm, sl*np.array(osm)+ic, color="#e74c3c", lw=1.5)
axes[0,1].set_title(f"Q-Q Plot (r={r:.3f})")
axes[0,1].set_xlabel("Theoretical quantiles"); axes[0,1].set_ylabel("Sample quantiles")
# 3. Scale-Location
axes[1,0].scatter(fitted, np.sqrt(np.abs(std_resid)), alpha=0.5, s=20, color="steelblue")
axes[1,0].set_xlabel("Fitted values"); axes[1,0].set_ylabel("sqrt|Standardised residuals|")
axes[1,0].set_title("Scale-Location")
# 4. Residuals vs Leverage
axes[1,1].scatter(leverage, std_resid, alpha=0.5, s=20, color="steelblue")
axes[1,1].axhline(0, color="grey", lw=0.8)
axes[1,1].set_xlabel("Leverage"); axes[1,1].set_ylabel("Standardised residuals")
axes[1,1].set_title("Residuals vs Leverage")
plt.suptitle("Regression Diagnostic Panel", y=1.01)
plt.tight_layout(); plt.show()

---
## Formal Tests

In [ ]:
# Heteroscedasticity
bp_lm, bp_p, bp_f, bp_fp = het_breuschpagan(resid, model.model.exog)
print(f"Breusch-Pagan test: LM={bp_lm:.3f}, p={bp_p:.4f}")
sig = "Heteroscedasticity detected" if bp_p < 0.05 else "No evidence of heteroscedasticity"
print(f"  {sig}")
# Normality of residuals
w_stat, sw_p = stats.shapiro(resid)
print(f"\nShapiro-Wilk normality: W={w_stat:.4f}, p={sw_p:.4f}")
nrm = "Non-normal residuals" if sw_p < 0.05 else "No evidence against normality"
print(f"  {nrm}")
# Autocorrelation
dw = durbin_watson(resid)
print(f"\nDurbin-Watson: {dw:.4f}  (2.0 = no autocorrelation; <1.5 or >2.5 = concern)")

---
## Diagnosing a Misspecified Model

In [ ]:
# Introduce heteroscedasticity deliberately
richness_mis = (28 - 0.035*elevation - 0.8*nitrate + 2.5*treatment
                + rng.normal(0, 0.015*elevation, n))
model_mis = sm.OLS(richness_mis, X).fit()
resid_mis  = model_mis.resid
fitted_mis = model_mis.fittedvalues
bp2_lm, bp2_p, _, _ = het_breuschpagan(resid_mis, model_mis.model.exog)
fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax, res, fit, title, pval in [
        (axes[0], resid,     fitted,     "Well-specified",   bp_p),
        (axes[1], resid_mis, fitted_mis, "Heteroscedastic",  bp2_p)]:
    ax.scatter(fit, res, alpha=0.5, s=20, color="steelblue")
    ax.axhline(0, color="#e74c3c", lw=1.5)
    ax.set_xlabel("Fitted"); ax.set_ylabel("Residuals")
    ax.set_title(f"{title}\nBreusch-Pagan p={pval:.4f}")
plt.tight_layout(); plt.show()
print("Fan-shaped residuals -> use WLS or log-transform y")

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.stem(range(n), cooks_d, markerfmt="C0o", linefmt="C0-",
        basefmt="k-", use_line_collection=True)
thresh = 4/n
ax.axhline(thresh, color="#e74c3c", lw=1.5, linestyle="--",
           label=f"Threshold 4/n={thresh:.3f}")
n_inf = (cooks_d > thresh).sum()
ax.set_xlabel("Observation index"); ax.set_ylabel("Cook's distance")
ax.set_title(f"Cook's Distance: {n_inf} high-influence observations")
ax.legend(); plt.tight_layout(); plt.show()
print(f"Top 3 by Cook's: indices {np.argsort(cooks_d)[-3:][::-1]}")
print("Investigate -- do not auto-remove")

---

## Common Pitfalls

**1. Skipping diagnostic plots because the R-squared looks good**  
A high R-squared can coexist with severely non-normal residuals, strong heteroscedasticity, or influential points driving the fit. R-squared is not a diagnostic. Always plot the four-panel diagnostic regardless of apparent model fit.

**2. Applying formal normality tests to large samples**  
Shapiro-Wilk detects trivially small deviations at large n. A p < 0.05 at n = 1000 may reflect a departure with no practical consequence. Use the Q-Q plot as the primary assessment; use formal tests as supplements at small n.

**3. Automatically removing influential points flagged by Cook's distance**  
High Cook's distance identifies leverage observations — they may be genuine extremes, data errors, or scientifically important cases. Always investigate before removing. Removing valid observations to improve fit is cherry-picking.

**4. Ignoring the Scale-Location plot and only checking Residuals vs Fitted**  
Residuals vs Fitted can appear flat even with heteroscedasticity if the variance pattern is subtle. The Scale-Location plot amplifies it by plotting sqrt of absolute standardised residuals.

**5. Not checking autocorrelation for time-ordered or spatially-indexed data**  
For observations collected over time or space, independence is often violated. Always run Durbin-Watson (or Moran's I for spatial data) when observations have a natural ordering.
---
*python_methods_library - Samantha McGarrigle*